In [8]:
#from Video_load_functions import generate_description
#from Video_load_functions import framing
import base64
import os
from openai import OpenAI
import pandas as pd 
import polars as pl 
#import moviepy

In [9]:
client = OpenAI(
    base_url=os.getenv("LMSTUDIO_BASE_URL", "http://localhost:1234/v1"),
    api_key=os.getenv("LMSTUDIO_API_KEY", "lm-studio")
)

In [10]:
def framing(name = "", source = "Videos/videoplayback.mp4", fps = 2, start_time = 0, end_time = 0):
    name = []
    start_time = start_time * fps
    end_time = end_time * fps
    with moviepy.VideoFileClip(source) as clip:
        for frame_idx in range(start_time, end_time):
            t = frame_idx/fps
            frame = clip.get_frame(t).astype("uint8")
            name.append(frame)
    return name
#This funtion is "deprecated" and is here just as a remeber

https://docs.pola.rs/api/python/stable/reference/dataframe/index.html

El tema se ha puesto serio que hay que sacar al hermano listo de pandas, no se pueden guardar tantos datos en un csv, así que un parquet es mejor opcion.

https://developers.openai.com/api/reference/python/resources/chat/subresources/completions/methods/create

In [11]:
def read_image(frames, prompt="What is in these images? Max response 20 words"):
    import base64
    if isinstance(frames, (bytes, bytearray)):
        frames = [frames] #This part creates a list only if frames are bytes, in order to not list a list 
    content = [{"type": "input_text", "text": prompt}]
    for frame in frames:
        image_b64 = base64.b64encode(frame).decode("utf-8")
        data_url = f"data:image/png;base64,{image_b64}"
        content.append({
            "type": "input_image",
            "image_url": data_url
            })

    response = client.responses.create(
        model="google/gemma-4-e4b",
        input=[{
            "role": "user",
            "content": content
        }]
    )
    return response.output_text

In [12]:
def generate_description(frame_list, min_init = 0, min_fin= 0):
    N1 = min_init 
    N2 = min_fin 
    #N1 = 17910
    #N2 = 18090 #test pourpouse
    minute_frame_list = []
    #Test sending 3 frames every time 
    for i in range(N1, N2, 3):
        batch = frame_list[i:i+3]
        descripcion = read_image(batch, test_promt)
        minute_frame_list.append(descripcion)
    return minute_frame_list

In [13]:
test_promt = """
Eres un experto en análisis visual de fútbol para El Clásico Real Madrid vs. FC Barcelona
Arriba a la izquierda se muestra el tiempo en formato MM:SS
## Instrucciones
Describe la situación en la imagen de forma breve (20-50 palabras): momento, acción clave, posiciones y táctica. Usa nombres en lugar del dorsal dorsal (e.g., Messi #10).
## Dorsales y Nombres (Alineaciones)
**Real Madrid (4-3-3)** jugadores de color blanco:
- #25: Diego López (portero)
- #15: Dani Carvajal (lateral der.)
- #4: Sergio Ramos (central)
- #3: Pepe (central)
- #12: Marcelo (lateral izq.)
- #14: Xabi Alonso (mediocentro)
- #19: Luka Modrić (interior der.)
- #22: Ángel Di María (interior izq.)
- #11: Gareth Bale (extremo der.)
- #9: Karim Benzema (delantero)
- #7: Cristiano Ronaldo (extremo izq.)

**Barcelona (4-3-3)** jugadores de color azul:
- #1: Víctor Valdés (portero)
- #22: Dani Alves (lateral der.)
- #3: Gerard Piqué (central)
- #18: Jordi Alba (lateral izq.)
- #14: Javier Mascherano (mediocentro defensivo)
- #6: Xavi Hernández (interior der.) 
- #8: Andrés Iniesta (interior izq.)
- #4: Cesc Fàbregas (mediapunta)
- #10: Lionel Messi (delantero der.)
- #11: Neymar (delantero izq.)
- #17: Pedro (extremo, entró después)"""

In [14]:
import polars as pl
First_time_frames = pl.read_parquet("data/first_time.parquet") 
#Arround 15-20 seconds in read(Javier Mac)


In [15]:
"""
r= [data for data in data_2["frame_col"]]
print(type(r[0]), type(r))
"""
#Testing in order to see the typer of the stored information to correct the loop
# So r is a list, and the first element of the list are the bytes

'\nr= [data for data in data_2["frame_col"]]\nprint(type(r[0]), type(r))\n'

In [16]:
"""
data_2 = First_time_frames.sample(1)
data_2 = data_2.to_pandas()
data_2["Description"] = [read_image(data) for data in data_2["frame_col"]]
print(data_2["Description"])
"""
#Test, converted to pandas as polars does not write colums in thes method. 

'\ndata_2 = First_time_frames.sample(1)\ndata_2 = data_2.to_pandas()\ndata_2["Description"] = [read_image(data) for data in data_2["frame_col"]]\nprint(data_2["Description"])\n'

In [17]:
"""
data_2 = First_time_frames.head(5)
batch_description_test = [read_image(data_2["frame_col"][i:i+3])
    for i in range(0, len(data_2), 3)
] 
""" 
#Test for batches, Processing more than 1 frame

'\ndata_2 = First_time_frames.head(5)\nbatch_description_test = [read_image(data_2["frame_col"][i:i+3])\n    for i in range(0, len(data_2), 3)\n] \n'

In [18]:
"""
data_2 = First_time_frames.head(5)
data_3 = data_2[:,0].to_list()
prueba = generate_description(data_3, 0, 5)
print(prueba)"""
#Test for the double function, generate_description batches and then read_image describes the situation

'\ndata_2 = First_time_frames.head(5)\ndata_3 = data_2[:,0].to_list()\nprueba = generate_description(data_3, 0, 5)\nprint(prueba)'

In [24]:
First_time_frames = pl.read_parquet("data/first_time.parquet", n_rows= 1300) 
First_5min_frames_list = First_time_frames[:,0].head(1250).to_list()
description_list = generate_description(First_5min_frames_list, min_init= 125, min_fin= 250)

In [28]:
import polars as pl
df = pl.DataFrame(description_list)
df.write_csv("data/Description_List_1min.csv")